In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/')
!cp "/content/drive/MyDrive/Colab Notebooks/Basura_Dataset.zip" /content/
!unzip -q /content/Basura_Dataset.zip -d /content/

In [ ]:
import os
import copy
from time import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import torchvision.models as models

class SafeImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes, class_to_idx = super().find_classes(directory)
        valid_classes = []
        for c in classes:
            class_path = os.path.join(directory, c)
            if os.path.isdir(class_path):
                has_images = any(f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')) for f in os.listdir(class_path))
                if has_images:
                    valid_classes.append(c)
        new_class_to_idx = {cls_name: i for i, cls_name in enumerate(sorted(valid_classes))}
        return sorted(valid_classes), new_class_to_idx

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
transformaciones_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
])

transformaciones_val_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset_entrenamiento = SafeImageFolder(root='Basura_Dataset/train', transform=transformaciones_train)

try:
    dataset_validacion = SafeImageFolder(root='Basura_Dataset/val', transform=transformaciones_val_test)
except FileNotFoundError:
    dataset_validacion = dataset_entrenamiento

try:
    dataset_test = SafeImageFolder(root='Basura_Dataset/test', transform=transformaciones_val_test)
except FileNotFoundError:
    dataset_test = dataset_entrenamiento

dataloader_entrenamiento = DataLoader(dataset_entrenamiento, batch_size=32, shuffle=True)
dataloader_validacion = DataLoader(dataset_validacion, batch_size=32, shuffle=False)
dataloader_test = DataLoader(dataset_test, batch_size=32, shuffle=False)

In [ ]:
modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_ftrs = modelo.fc.in_features
modelo.fc = nn.Linear(num_ftrs, 7)

modelo = modelo.to(device)

criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.0005)

In [ ]:
epocas = 50
paciencia = 5
mejor_perdida_val = float('inf')
mejor_epoca = 0
mejor_estado = None
epocas_sin_mejora = 0

lista_perdida_train = []
lista_perdida_val = []
lista_acc_train = []
lista_acc_val = []

comienzo = time()

for epoca in range(epocas):
    modelo.train()
    perdida_total_epoca = 0.0
    y_real_train = []
    y_pred_train = []

    for imagenes, etiquetas in dataloader_entrenamiento:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        predicciones = modelo(imagenes)
        perdida = criterio(predicciones, etiquetas)
        perdida.backward()
        optimizador.step()

        perdida_total_epoca += perdida.item()
        y_pred_train.extend(predicciones.argmax(dim=1).cpu().numpy())
        y_real_train.extend(etiquetas.cpu().numpy())

    perdida_train_promedio = perdida_total_epoca / len(dataloader_entrenamiento)
    acc_train = accuracy_score(y_real_train, y_pred_train)

    modelo.eval()
    perdida_val_total = 0.0
    y_real_val = []
    y_pred_val = []

    with torch.no_grad():
        for imagenes, etiquetas in dataloader_validacion:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            predicciones = modelo(imagenes)
            perdida = criterio(predicciones, etiquetas)

            perdida_val_total += perdida.item()
            y_pred_val.extend(predicciones.argmax(dim=1).cpu().numpy())
            y_real_val.extend(etiquetas.cpu().numpy())

    perdida_val_promedio = perdida_val_total / len(dataloader_validacion)
    acc_val = accuracy_score(y_real_val, y_pred_val)

    lista_perdida_train.append(perdida_train_promedio)
    lista_perdida_val.append(perdida_val_promedio)
    lista_acc_train.append(acc_train)
    lista_acc_val.append(acc_val)

    print(f"Época [{epoca+1:2d}/{epocas}] | Train Loss: {perdida_train_promedio:.4f} | Train Acc: {acc_train:.4f} | Val Loss: {perdida_val_promedio:.4f} | Val Acc: {acc_val:.4f}")

    if perdida_val_promedio < mejor_perdida_val:
        mejor_perdida_val = perdida_val_promedio
        mejor_epoca = epoca + 1
        mejor_estado = copy.deepcopy(modelo.state_dict())
        epocas_sin_mejora = 0
        print("   -> Mejora en validación: guardamos este modelo")
    else:
        epocas_sin_mejora += 1
        print(f"   -> Sin mejora ({epocas_sin_mejora}/{paciencia})")
        if epocas_sin_mejora >= paciencia:
            print(f"\nParada temprana en la época {epoca+1}: la mejor fue la {mejor_epoca} (Val Loss: {mejor_perdida_val:.4f})")
            break

tiempo_entrenamiento = time() - comienzo
print(f"\n¡Entrenamiento completado! Tiempo total: {tiempo_entrenamiento:.2f} segundos")

In [ ]:
modelo.load_state_dict(mejor_estado)

modelo.eval()
y_real_test = []
y_pred_test = []

with torch.no_grad():
    for imagenes, etiquetas in dataloader_test:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        predicciones = modelo(imagenes)
        y_pred_test.extend(predicciones.argmax(dim=1).cpu().numpy())
        y_real_test.extend(etiquetas.cpu().numpy())

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(lista_perdida_train) + 1), lista_perdida_train, label="Train Loss")
plt.plot(range(1, len(lista_perdida_val) + 1), lista_perdida_val, label="Val Loss")
plt.axvline(mejor_epoca, color="red", linestyle="--")
plt.legend()
plt.show()

cm = confusion_matrix(y_real_test, y_pred_test)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset_test.classes, yticklabels=dataset_test.classes)
plt.ylabel('Etiqueta Real')
plt.xlabel('Predicción')
plt.show()

print(classification_report(y_real_test, y_pred_test, target_names=dataset_test.classes))